In [5]:
!pip install -q gradio openpyxl plotly scikit-learn

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import gradio as gr

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [7]:
file = r"/content/Un Student Experience & Engagement Survey  (Responses) (1).xlsx"

df = pd.read_excel(file)

In [8]:
df = df.dropna(how="all")
df = df.dropna(axis=1, how="all")
df = df.drop_duplicates()
df.columns = df.columns.str.strip()

In [9]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.columns.tolist())

Rows: 145
Columns: 18
['Timestamp', 'Which program/department are you currently studying?', 'Which semester are you currently studying?', 'How satisfied are you with the quality and availability of classroom facilities?', 'How satisfied are you with faculty interaction, guidance, and support?', 'How satisfied are you with the availability and quality of campus Wi-Fi?', 'How satisfied are you with cafeteria food, cleanliness, and service?', 'How satisfied are you with library facilities, resources, and study environment?', 'How satisfied are you with sports facilities and opportunities?', 'How satisfied are you with student clubs and extracurricular activities?', 'How satisfied are you with technical events, workshops, seminars, and hackathons?', 'How satisfied are you with campus cleanliness and maintenance?', 'How satisfied are you with safety and security on campus?', 'How satisfied are you with university transportation facilities?', 'How actively do you participate in campus activi

In [10]:
department_col = None

for col in df.columns:
    name = col.lower()

    if (
        "department" in name
        or "program" in name
        or "course" in name
        or "degree" in name
    ):
        department_col = col
        break

if department_col is None:
    df["Department"] = "All Students"
    department_col = "Department"

feature_words = {
    "Classrooms": ["classroom"],
    "Faculty": ["faculty"],
    "Wi-Fi": ["wifi", "wi-fi", "wi fi", "internet"],
    "Cafeteria": ["cafeteria", "canteen"],
    "Library": ["library"],
    "Sports": ["sports", "sport"],
    "Clubs": ["club"],
    "Tech Events": ["technical event", "technical events", "tech event"],
    "Cleanliness": ["cleanliness", "clean"],
    "Safety": ["safety", "safe"],
    "Transport": ["transport", "transportation", "bus"]
}

features = {}

for short_name, words in feature_words.items():
    for col in df.columns:
        col_name = col.lower()

        if any(word in col_name for word in words):
            values = pd.to_numeric(df[col], errors="coerce")

            if values.notna().sum() > 2:
                df[short_name] = values.fillna(values.median())
                features[short_name] = short_name
                break

feature_names = list(features.keys())

print("Department:", department_col)
print("Features:", feature_names)
print("Number of features:", len(feature_names))

Department: Which program/department are you currently studying?
Features: ['Classrooms', 'Faculty', 'Wi-Fi', 'Cafeteria', 'Library', 'Sports', 'Clubs', 'Tech Events', 'Cleanliness', 'Safety', 'Transport']
Number of features: 11


In [11]:
df["Average Rating"] = df[feature_names].mean(axis=1)

df["Happiness"] = (
    (df["Average Rating"] - 1) / 4 * 100
).round(2)

def get_category(score):
    if score >= 80:
        return "Very Happy"
    elif score >= 60:
        return "Happy"
    elif score >= 40:
        return "Neutral"
    else:
        return "Needs Improvement"

df["Category"] = df["Happiness"].apply(get_category)

print(df[["Average Rating", "Happiness", "Category"]].head())

   Average Rating  Happiness           Category
0        5.000000     100.00         Very Happy
1        3.818182      70.45              Happy
2        2.545455      38.64  Needs Improvement
3        3.818182      70.45              Happy
4        2.363636      34.09  Needs Improvement


In [12]:
print("Total Students:", len(df))
print("Average Rating:", round(df["Average Rating"].mean(), 2))
print("Average Happiness:", round(df["Happiness"].mean(), 2))

facility_average = df[feature_names].mean().sort_values()

fig = px.bar(
    facility_average,
    title="⭐ Average Facility Satisfaction",
    color=facility_average,
    color_continuous_scale="Turbo",
    text_auto=".2f"
)

fig.update_yaxes(range=[0, 5])
fig.show()

Total Students: 145
Average Rating: 3.6
Average Happiness: 65.08


In [13]:
X = df[feature_names]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

cluster_average = (
    df.groupby("Cluster")["Happiness"]
    .mean()
    .sort_values()
)

cluster_order = cluster_average.index.tolist()

cluster_names = {
    cluster_order[0]: "Low Satisfaction",
    cluster_order[1]: "Moderate Satisfaction",
    cluster_order[2]: "High Satisfaction"
}

df["Cluster Name"] = df["Cluster"].map(cluster_names)

print(df["Cluster Name"].value_counts())

Cluster Name
Moderate Satisfaction    70
High Satisfaction        59
Low Satisfaction         16
Name: count, dtype: int64


In [14]:
cluster_data = (
    df["Cluster Name"]
    .value_counts()
    .reset_index()
)

cluster_data.columns = [
    "Group",
    "Students"
]

fig = px.bar(
    cluster_data,
    x="Group",
    y="Students",
    color="Group",
    text="Students",
    title="🔵 Student Satisfaction Groups"
)

fig.show()

In [15]:
df["Satisfaction"] = np.where(
    df["Happiness"] >= 60,
    "Satisfied",
    "Not Satisfied"
)

X = df[feature_names]
y = df["Satisfaction"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

tree = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree.fit(X_train, y_train)

prediction = tree.predict(X_test)

accuracy = accuracy_score(
    y_test,
    prediction
)

print(
    "Decision Tree Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Decision Tree Accuracy: 93.1 %


In [16]:
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": tree.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

        Feature  Importance
6         Clubs    0.485482
3     Cafeteria    0.178657
9        Safety    0.096337
4       Library    0.087380
10    Transport    0.077304
0    Classrooms    0.035389
1       Faculty    0.027962
8   Cleanliness    0.011490
2         Wi-Fi    0.000000
5        Sports    0.000000
7   Tech Events    0.000000


In [17]:
dashboard_data = df.copy()

dashboard_data["Department"] = (
    dashboard_data[department_col].astype(str)
)

departments = sorted(
    dashboard_data["Department"].unique()
)

print(departments)

['B.Tech', 'BCA', 'BSc', 'M.Tech', 'MCA', 'MSc']


In [18]:
suggestions = {
    "Classrooms": "Improve classroom facilities, seating, lighting and ventilation.",
    "Faculty": "Increase faculty-student interaction, mentoring and doubt-clearing sessions.",
    "Wi-Fi": "Improve Wi-Fi speed, coverage and reliability across campus.",
    "Cafeteria": "Improve food quality, hygiene, seating and menu variety.",
    "Library": "Add more books, digital resources, study spaces and longer library hours.",
    "Sports": "Improve sports facilities, equipment and student participation.",
    "Clubs": "Conduct more club activities and encourage student participation.",
    "Tech Events": "Organize more hackathons, workshops and technical seminars.",
    "Cleanliness": "Increase cleaning frequency in classrooms, washrooms and common areas.",
    "Safety": "Improve security, lighting, CCTV monitoring and emergency support.",
    "Transport": "Improve bus availability, timing, routes and transportation communication."
}

In [19]:
def dashboard(department):

    if department == "All":
        data = dashboard_data.copy()
    else:
        data = dashboard_data[
            dashboard_data["Department"] == department
        ]

    students = len(data)

    average_rating = round(
        data["Average Rating"].mean(), 2
    )

    happiness = round(
        data["Happiness"].mean(), 2
    )

    result = f"""
# 🎓 Smart Campus Analytics

| Metric | Result |
|---|---:|
| 👨‍🎓 Students | **{students}** |
| ⭐ Average Rating | **{average_rating}/5** |
| 😊 Happiness Index | **{happiness}/100** |
| 🏫 Department | **{department}** |
"""

    facility = (
        data[feature_names]
        .mean()
        .sort_values()
    )

    graph1 = px.bar(
        x=facility.values,
        y=facility.index,
        orientation="h",
        title="Facility Satisfaction",
        text=[f"{x:.2f}" for x in facility.values]
    )

    graph1.update_traces(
        marker_color="#2F75B5",
        textposition="outside"
    )

    graph1.update_layout(
        xaxis_title="Average Rating",
        yaxis_title="",
        xaxis=dict(range=[0, 5]),
        plot_bgcolor="#F2F2F2",
        paper_bgcolor="white",
        font=dict(color="#1F1F1F")
    )

    category_data = (
        data["Category"]
        .value_counts()
        .reset_index()
    )

    category_data.columns = [
        "Category",
        "Students"
    ]

    graph2 = px.pie(
        category_data,
        names="Category",
        values="Students",
        title="Student Happiness Distribution",
        hole=0.45
    )

    graph2.update_traces(
        textposition="inside",
        textinfo="percent+label"
    )

    graph2.update_layout(
        paper_bgcolor="white",
        font=dict(color="#1F1F1F")
    )

    department_data = (
        dashboard_data
        .groupby("Department")["Happiness"]
        .mean()
        .reset_index()
    )

    graph3 = px.bar(
        department_data,
        x="Department",
        y="Happiness",
        title="Department-wise Happiness Index",
        text_auto=".1f"
    )

    graph3.update_traces(
        marker_color="#00A6A6"
    )

    graph3.update_layout(
        yaxis=dict(range=[0, 100]),
        xaxis_title="Department",
        yaxis_title="Happiness Index",
        plot_bgcolor="#F2F2F2",
        paper_bgcolor="white",
        font=dict(color="#1F1F1F")
    )

    concerns = (
        facility
        .sort_values()
        .head(10)
        .sort_values(ascending=True)
    )

    graph4 = px.bar(
        x=concerns.values,
        y=concerns.index,
        orientation="h",
        title="Top Student Concerns",
        text=[f"{x:.2f}" for x in concerns.values]
    )

    graph4.update_traces(
        marker_color="#ED7D31",
        textposition="outside"
    )

    graph4.update_layout(
        xaxis_title="Average Rating",
        yaxis_title="",
        xaxis=dict(range=[0, 5]),
        plot_bgcolor="#F2F2F2",
        paper_bgcolor="white",
        font=dict(color="#1F1F1F")
    )

    action_data = []

    for feature in feature_names:

        avg = round(
            data[feature].mean(), 2
        )

        if avg < 2.5:
            priority = "🔴 High"
        elif avg < 3.5:
            priority = "🟠 Medium"
        else:
            priority = "🟢 Low"

        action_data.append({
            "Area": feature,
            "Rating": avg,
            "Priority": priority,
            "Suggestion": suggestions.get(
                feature,
                "Continue monitoring this area."
            )
        })

    action_df = pd.DataFrame(action_data)

    action_df = action_df.sort_values(
        "Rating"
    )

    action_text = """
# 💡 Action Plan

The following recommendations are generated from student satisfaction ratings.

🔴 **High Priority** → Immediate action

🟠 **Medium Priority** → Improvement recommended

🟢 **Low Priority** → Maintain current performance

"""

    for _, row in action_df.iterrows():

        action_text += f"""
### {row['Area']}

**Rating:** {row['Rating']}/5
**Priority:** {row['Priority']}

**Recommendation:** {row['Suggestion']}

---
"""

    return (
        result,
        graph1,
        graph2,
        graph3,
        graph4,
        action_text
    )

In [20]:
def analyze_student(
    department,
    classroom,
    faculty,
    wifi,
    cafeteria,
    library,
    sports,
    clubs,
    events,
    cleanliness,
    safety,
    transport
):

    ratings = [
        classroom,
        faculty,
        wifi,
        cafeteria,
        library,
        sports,
        clubs,
        events,
        cleanliness,
        safety,
        transport
    ]

    average = np.mean(ratings)

    happiness = (
        (average - 1) / 4 * 100
    )

    happiness = round(
        happiness,
        2
    )

    if happiness >= 80:
        status = "Very Happy 😊"
    elif happiness >= 60:
        status = "Happy 🙂"
    elif happiness >= 40:
        status = "Neutral 😐"
    else:
        status = "Needs Improvement ⚠️"

    names = [
        "Classrooms",
        "Faculty",
        "Wi-Fi",
        "Cafeteria",
        "Library",
        "Sports",
        "Clubs",
        "Tech Events",
        "Cleanliness",
        "Safety",
        "Transport"
    ]

    low_areas = []

    for name, rating in zip(names, ratings):

        if rating <= 2:
            low_areas.append(name)

    if len(low_areas) == 0:

        recommendation = (
            "🎉 No major problem found. "
            "Continue maintaining the facilities."
        )

    else:

        recommendation = (
            "⚠️ Areas needing improvement: "
            + ", ".join(low_areas)
        )

    return f"""
# 👨‍🎓 Student Analysis Result

### 🏫 Department
**{department}**

### ⭐ Average Rating
**{average:.2f}/5**

### 😊 Happiness Index
# **{happiness}/100**

### 📊 Student Status
# **{status}**

### 💡 Recommendation
{recommendation}
"""

In [21]:
with gr.Blocks(
    title="Smart Campus Analytics"
) as app:

    gr.Markdown(
        """
# 🎓 Smart Campus Analytics

### Student Experience & Engagement Analytics

**Data Cleaning → EDA → K-Means → Decision Tree → Dashboard → Action Plan**
"""
    )

    with gr.Tab("📊 Live Dashboard"):

        gr.Markdown(
            "## 📊 Interactive Student Satisfaction Dashboard"
        )

        department_input = gr.Dropdown(
            choices=["All"] + departments,
            value="All",
            label="🏫 Select Department"
        )

        show_button = gr.Button(
            "📊 Show Dashboard",
            variant="primary"
        )

        dashboard_result = gr.Markdown()

        graph1 = gr.Plot()

        graph2 = gr.Plot()

        graph3 = gr.Plot()

        graph4 = gr.Plot()

        action_output = gr.Markdown()

        show_button.click(
            dashboard,
            inputs=department_input,
            outputs=[
                dashboard_result,
                graph1,
                graph2,
                graph3,
                graph4,
                action_output
            ]
        )

    with gr.Tab("👨‍🎓 Student Analysis"):

        gr.Markdown(
            """
## 👨‍🎓 Enter Student Experience

1 = Very Poor
2 = Poor
3 = Average
4 = Good
5 = Excellent
"""
        )

        student_department = gr.Dropdown(
            choices=departments,
            value=departments[0],
            label="🏫 Department"
        )

        classroom = gr.Slider(
            1, 5, 3, step=1,
            label="Classrooms"
        )

        faculty = gr.Slider(
            1, 5, 3, step=1,
            label="Faculty Interaction"
        )

        wifi = gr.Slider(
            1, 5, 3, step=1,
            label="Wi-Fi"
        )

        cafeteria = gr.Slider(
            1, 5, 3, step=1,
            label="Cafeteria"
        )

        library = gr.Slider(
            1, 5, 3, step=1,
            label="Library"
        )

        sports = gr.Slider(
            1, 5, 3, step=1,
            label="Sports"
        )

        clubs = gr.Slider(
            1, 5, 3, step=1,
            label="Clubs"
        )

        events = gr.Slider(
            1, 5, 3, step=1,
            label="Technical Events"
        )

        cleanliness = gr.Slider(
            1, 5, 3, step=1,
            label="Cleanliness"
        )

        safety = gr.Slider(
            1, 5, 3, step=1,
            label="Safety"
        )

        transport = gr.Slider(
            1, 5, 3, step=1,
            label="Transportation"
        )

        analyze_button = gr.Button(
            "🔍 Analyze Student",
            variant="primary"
        )

        student_result = gr.Markdown()

        analyze_button.click(
            analyze_student,
            inputs=[
                student_department,
                classroom,
                faculty,
                wifi,
                cafeteria,
                library,
                sports,
                clubs,
                events,
                cleanliness,
                safety,
                transport
            ],
            outputs=student_result
        )

    with gr.Tab("📋 Cleaned Data"):

        gr.Markdown(
            "## 📋 Cleaned Survey Data"
        )

        gr.Dataframe(
            dashboard_data.head(100),
            interactive=False
        )

    with gr.Tab("🤖 Data Mining"):

        gr.Markdown(
            f"""
# 🤖 Data Mining Results

### 🔵 K-Means Clustering

**3 student groups**

- Low Satisfaction
- Moderate Satisfaction
- High Satisfaction

### 🌳 Decision Tree

**Prediction:** Satisfied / Not Satisfied

**Accuracy:** {accuracy * 100:.2f}%
"""
        )

        gr.Dataframe(
            importance,
            interactive=False
        )

    with gr.Tab("💡 Action Plan"):

        gr.Markdown(
            """
# 💡 Campus Improvement Action Plan

The system identifies low-rated areas and
provides practical improvement suggestions.
"""
        )

        action_table = pd.DataFrame({
            "Area": feature_names,
            "Rating": [
                round(df[x].mean(), 2)
                for x in feature_names
            ],
            "Suggestion": [
                suggestions.get(
                    x,
                    "Continue monitoring."
                )
                for x in feature_names
            ]
        })

        action_table = action_table.sort_values(
            "Rating"
        )

        gr.Dataframe(
            action_table,
            interactive=False
        )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8455fccd7dcc7225c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
